# Spell Checker



Una de las primeras aplicaciones de los modelos de IA en problemas de lenguaje (además de los clasificadores de spam) los modelos fueron los correctores ortotipográficos. Existen diferentes formas de atacar ese problema (IA tradicional, Machine Learning, Deep Learning...).

Desde la perspectiva del Machine learning supervisado necesitaríamos un dataset (o corpus) de palabras con errores y su correspondiente palabra correcta como target. Por ejemplo, podríamos generar el siguiente corpus de palabras a partir de 100 años de soledad


## Dataset

In [1]:
import re
import requests


url = "https://gist.githubusercontent.com/ismaproco/6781d297ee65c6a707cd3c901e87ec56/raw/20d3520cd7c53d99215845375b1dca16ac827bd7/gabriel_garcia_marquez_cien_annos_soledad.txt"
text = requests.get(url).text.lower()  # Convertimos a minúsculas

# Tokenizamos palabras (solo a-z)
words = re.findall(r'\b[a-záéíóúñü]+\b', text)

# Quitamos duplicados para tener vocabulario
vocab = list(set(words))
print("Número de palabras únicas en el vocabulario:", len(vocab))
print(words[0:30])

Número de palabras únicas en el vocabulario: 15692
['gabriel', 'garcía', 'márquez', 'cien', 'años', 'de', 'soledad', 'editado', 'por', 'ediciones', 'la', 'cueva', 'para', 'j', 'omi', 'garcía', 'ascot', 'y', 'maría', 'luisa', 'elio', 'cien', 'años', 'de', 'soledad', 'gabriel', 'garcía', 'márquez', 'muchos', 'años']


## Canal de ruido

A partir del lenguaje natural podemos obtener palabras, pero cuesta mucho buscar/generar errores manualmente. Para este problema en el NLP se usaba una técnica que se denomina "canal de ruido" Que permite generar un dataset artificial a partir de una función que genera errores

In [ ]:
import numpy as np

# Mapa simple QWERTY (fila, columna)
keyboard = {
    'q': (0,0), 'w': (0,1), 'e': (0,2), 'r': (0,3), 't': (0,4), 'y': (0,5), 'u': (0,6), 'i': (0,7), 'o': (0,8), 'p': (0,9),
    'a': (1,0), 's': (1,1), 'd': (1,2), 'f': (1,3), 'g': (1,4), 'h': (1,5), 'j': (1,6), 'k': (1,7), 'l': (1,8), 'ñ': (1,9),  "'" : (1,10),
    'z': (2,0), 'x': (2,1), 'c': (2,2), 'v': (2,3), 'b': (2,4), 'n': (2,5), 'm': (2,6)
}

> **NOTA**: Usamos esta forma de "vectorizar" los caracteres (sin tokenizar y vectorizar como hemos visto antes) porque es conveniente para representar las letras en un teclado. Desde una perspectiva mecánica, las "features" de una letra en el teclado es su posición en términos de fila y columna

In [ ]:
import random

def generate_typo(word, error_prob=0.1):

    # Elegimos aleatoriamente la posición del error
    pos = random.randint(0, len(word)-1)

    # obtener posición
    x, y = keyboard[word[pos]]

    # letras cercanas
    neighbors = [c for c in keyboard if (keyboard[c][0] in [x-1,x+1]) and (keyboard[c][1] in [y-1,y+1])]

    # Elegir una letra
    wrong_character = random.choice(neighbors)

    # Reemplazamos el caracter en la posición elegida
    word = word[:pos] + wrong_character + word[pos+1:]

    return word

In [ ]:
generate_typo('hola')

'hopa'

El objetivo de este ejercicio es usar la función generate_typo para hacer un modelo de aprendizaje supervisado que aprenda a corregir errores

<br>

---

### Preprocesado de texto

Para poder corregir errores de tilde, representamos la tilde como un caracter separado a la vocal

In [ ]:
import unicodedata

def preprocess_text(text):
    """
    Convierte caracteres acentuados en letra base + apóstrofe.
    Ejemplo: 'í' -> "i'"
    """

    text = text.lower()
    text= text.replace("ñ","___")

    # Normalizamos el string a NFD (decomposición)
    text = unicodedata.normalize('NFD', text)

    # Reemplazamos cualquier diacrítico (acento) por apostrofe
    text = ''.join("'" if unicodedata.combining(c) else c for c in text)

    text = text.replace("___","ñ")
    return text

# Ejemplos de uso
print(preprocess_text("buendía"))
print(preprocess_text("hola"))
print(preprocess_text("años"))

buendi'a
hola
años


In [ ]:
words = [preprocess_text(word) for word in words]

<br>

### Generación de errores

La tarea de nuestro modelo es "corregir palabras mal escritas" por tanto necesitamos que aprenda, a partir de una palabra mal escrita cual ese la palabra correcta (target)

In [ ]:
import pandas as pd

corpus = pd.DataFrame(
    {
        "typo": [generate_typo(word) for word in words],
        "word": words,
    }
)
corpus

,typo,word
0,gabdiel,gabriel
1,gwrci'a,garci'a
2,mw'rquez,ma'rquez
3,clen,cien
4,añoq,años
...,...,...
139129,xvj,xvi
139130,xvji,xvii
139131,xvlii,xviii
139132,xjx,xix


<br>

## Limpieza de datos

Eliminamos los errores que dan lugar a una palabra que esté bien escrita. Además también tenemos en cuenta solo palabras de más de 1 caracter

In [ ]:
corpus = corpus[
    corpus.apply(lambda row: row.typo not in vocab, axis=1)
][
    corpus.word.str.len() > 1 # Cogemos palabras de más de una letra
][
    corpus.word.str.len() < 15 # No cogemos palabras de más de 15 letras
].reset_index(drop=True)

corpus

/tmp/ipython-input-774576386.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[
/tmp/ipython-input-774576386.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[


,typo,word
0,gabdiel,gabriel
1,gwrci'a,garci'a
2,mw'rquez,ma'rquez
3,clen,cien
4,añoq,años
...,...,...
125717,xvj,xvi
125718,xvji,xvii
125719,xvlii,xviii
125720,xjx,xix


### Vectorización

Añadimos el token de "fin de palabra" al keyboard. Esto es algo relativo al modelo de machine learning. Le damos un valor muy distinto al resto de letras para diferenciarlo

In [ ]:
keyboard["</s>"] = (99,99)

Vectorizar consiste en atribuir features numéricas a un texto. En este caso la vectorización la hacemos a partir de la posición de las letras en vez de seguir un acercamiento más probabilístico como n-gramas y count vectorizer

In [ ]:
MAX_LEN = 15  # el número de features tiene que ser fijo así que definimos como "15" el número máximo de letras que puede tener una palabra

def word_to_vector(word):
    vec = []

    for i in range(MAX_LEN):
        if len(word) > i:
            vec = vec + list(keyboard[word.lower()[i]])
        else:
            vec = vec + list(keyboard["</s>"])

    # truncar si es más largo
    return np.array(vec[:MAX_LEN*2])

word_to_vector("hola")

array([ 1,  5,  0,  8,  1,  8,  1,  0, 99, 99, 99, 99, 99, 99, 99, 99, 99,
       99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99])

### Dataset

Para que el corpus de palabras nos sirva como dataset de entrenamiento guardamos las features en un dataframe

In [ ]:
dataset = pd.DataFrame(
    [word_to_vector(word) for word in corpus.typo]
).assign(word=corpus.word
).assign(typo=corpus.typo
).set_index('typo')

dataset

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,word
typo,,,,,,,,,,,,,,,,,,,,,
gabdiel,1,4,1,0,2,4,1,2,0,7,...,99,99,99,99,99,99,99,99,99,gabriel
gwrci'a,1,4,0,1,0,3,2,2,0,7,...,99,99,99,99,99,99,99,99,99,garci'a
mw'rquez,2,6,0,1,1,10,0,3,0,0,...,99,99,99,99,99,99,99,99,99,ma'rquez
clen,2,2,1,8,0,2,2,5,99,99,...,99,99,99,99,99,99,99,99,99,cien
añoq,1,0,1,9,0,8,0,0,99,99,...,99,99,99,99,99,99,99,99,99,años
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
xvj,2,1,2,3,1,6,99,99,99,99,...,99,99,99,99,99,99,99,99,99,xvi
xvji,2,1,2,3,1,6,0,7,99,99,...,99,99,99,99,99,99,99,99,99,xvii
xvlii,2,1,2,3,1,8,0,7,0,7,...,99,99,99,99,99,99,99,99,99,xviii


<br>

## Model

### train test split

In [ ]:
from sklearn.model_selection import train_test_split

X = dataset.drop(columns="word")
y = dataset.word

X_train, X_test, y_train, y_test = train_test_split(X,y)

### Train

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(
    n_neighbors=10,
    weights='distance',
    p=1
)

model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=10, p=1, weights='distance')

In [ ]:
# Ver qué vecinos usa el modelo para la predicción
vec_buenria = word_to_vector(preprocess_text("buenría"))
distances, indices = model.kneighbors([vec_buenria])

for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{i+1}. Distancia: {dist:.2f}, Palabra: {repr(y_train.iloc[idx])}")

1. Distancia: 0.00, Palabra: buendi'a
2. Distancia: 0.00, Palabra: buendi'a
3. Distancia: 0.00, Palabra: buendi'a
4. Distancia: 0.00, Palabra: buendi'a
5. Distancia: 0.00, Palabra: buendi'a
6. Distancia: 0.00, Palabra: buendi'a
7. Distancia: 0.00, Palabra: buendi'a
8. Distancia: 2.00, Palabra: buendi'a
9. Distancia: 2.00, Palabra: buendi'a
10. Distancia: 2.00, Palabra: buendi'a


### Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8696827972383953


In [ ]:
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

         a'gil       0.00      0.00      0.00         1
       a'mbito       0.80      1.00      0.89         4
        a'nimo       1.00      0.33      0.50         3
       a'nimos       1.00      1.00      1.00         1
       a'rabes       1.00      1.00      1.00         6
        a'rbol       1.00      1.00      1.00         1
       a'ridas       0.00      0.00      0.00         1
        a'rido       1.00      1.00      1.00         1
       a'rmate       0.00      0.00      0.00         1
       a'spera       0.00      0.00      0.00         1
       a'spero       0.50      1.00      0.67         1
      a'speros       0.00      0.00      0.00         1
         aaaay       0.00      0.00      0.00         1
    abandonaba       0.17      1.00      0.29         1
   abandonaban       0.40      1.00      0.57         2
    abandonada       0.00      0.00      0.00         3
   abandonadas       0.00      0.00      0.00  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

<br>

## Código de Spell Checker

Como hemos dicho alguna vez, la IA no solo es el modelo y los datos, también es el código que lo usa. Aquí podemos ver cómo sería un código funcional que realmente corrigiese la palabra más allá de simplemente predecir

In [ ]:
def spell_checker(word):

    if word in vocab: # Si la palabra existe en el vocabulario no se corrige
        return word

     # Preprocesamos el texto
    processed_word = preprocess_text(word)

    # Lo convertimos a vector
    vec = word_to_vector(processed_word)

    # Predecimos la palabra
    new_word = model.predict([vec])[0]

    # Calculamos la probabilidad de la predicción
    proba =  model.predict_proba([vec]).max()

    # Si el modelo no da una probabilidad suficientemente alta no corregimos la palabra
    if proba <0.5:
        return word


    # Deshacemos el preprocesamiento
    replacements = {
        "a'": "á",
        "e'": "é",
        "i'": "í",
        "o'": "ó",
        "u'": "ú",
        "n'": "ñ",
    }

    for k, v in replacements.items():
        new_word = new_word.replace(k, v)

    # Copiamos capitalización
    new_word = list(new_word)
    for char in word:
        if char.isupper():
            new_word[0] = new_word[0].upper()

    return "".join(new_word)

In [ ]:
spell_checker("Buenría")

'Buendía'